# 18.1 教師能提供什麼學習訊號？


同樣回答「4」，有一位教師把80%給4，另一位只給40%。想讓較小模型學它們的做法，這兩份指導應該相同嗎？教師究竟要交給學生什麼？最容易取得的是完整回答文字；若能取得模型每個位置的候選機率，還能提供「最終沒選的候選有多接近」。這兩種訊號包含的資訊不同，不能只因它們都來自教師就使用同一個誤差公式。蒸餾（distillation）指讓學生透過教師提供的訊號學習，教師本身不一定是真值。

[下一字分數](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)常叫logits，softmax把它們轉成總和為一的機率分布；[SFT](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)則用回答文字作目標，讓模型增加每個示範答案token的機率。只能看到教師生成文字的方式常叫黑盒蒸餾；可讀取教師分布的方式常叫白盒蒸餾。這些名字描述能拿到的訊號，不表示教師是不是可靠。

對問題「2+2=?」，教師可能回答「4」。假設候選是 `[4,3,5]`，分布 `[0.8,0.15,0.05]`的第一名是4，`[0.4,0.35,0.25]`的第一名也相同。只取得「4」無法分辨這兩種情況：它沒有記錄第二名與第一名的差距，也沒有記錄5的相對位置。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

candidates = ["4", "3", "5"]
probabilities = [torch.tensor([0.8, 0.15, 0.05]), torch.tensor([0.4, 0.35, 0.25])]
for prob in probabilities:
    answer = candidates[prob.argmax().item()]
    print("回答", answer, "分布", prob.tolist())

輸入是人工指定的兩個分布，用來隔離資訊差別，不是呼叫真的教師。`argmax`選最大機率索引，再查候選文字，兩行回答都應為4；分布欄卻不同。實際回答是一串token，黑盒資料保存整串文字；白盒通常要保存或即時讀取每個有效回答位置的一整列詞表分布，資料量和計算需求可能更大。

取得文字後，可由學生自己的tokenizer重新切分，所以黑盒方式能搭配不同詞表。逐欄對機率則必須確保兩邊同一欄代表同一個token，這是[詞表對齊](https://birdhackor.github.io/tiny-perceptron-vlm/18.6.html)會處理的問題。白盒能提供更多候選關係，也可能把教師的錯誤偏好一起交給學生；多一份資訊不等於保證更好。

練習將程式只改成 `print("回答", answer)`，先預測兩行完全相同，再執行核對。接著問自己：還能從留下的文字還原0.15、0.35這些數字嗎？答案不能，因為不同分布映成了同一個最大候選。這會決定後續哪些蒸餾方法有足夠輸入，哪些只能使用文字答案。

<details>
<summary>選讀：訊號分類、教師來源與方法範圍</summary>

[MiniLLM原文的導論](https://arxiv.org/abs/2306.08543v6)也按可取得的文字或分布區分這兩類訊號。本章的正式對照使用普通CE與固定前文上的forward KL，沒有採用該論文的reverse KL或policy-gradient方法。教師來源則是[直接SFT](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)、[條件式風格](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.5)與[MoE的固定top-2支線](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.8)，都載入實際訓練權重。它們的完整檔案SHA、架構與步數保存在[蒸餾實報的`teacher_provenance`](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/distillation.json)。

</details>
